In [1]:
import pandas as pd
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", None)

In [2]:
rfm = pd.read_csv("../data/processed/customer_rfm.csv")

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,48,1,77183.60
1,12347.0,39,2,1187.18
2,12348.0,41,2,1120.24
3,12350.0,32,1,334.40
4,12352.0,5,3,1281.15


In [3]:
X = rfm[["Recency", "Frequency", "Monetary"]]

X.head()

,Recency,Frequency,Monetary
0,48,1,77183.60
1,39,2,1187.18
2,41,2,1120.24
3,32,1,334.40
4,5,3,1281.15


In [4]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [5]:
X_scaled[:5]

array([[ 2.38620911e-01, -4.11767783e-01,  2.41297671e+01],
       [-6.01298380e-02, -1.22172859e-02,  7.46349451e-02],
       [ 6.25921727e-03, -1.22172859e-02,  5.34464398e-02],
       [-2.92491532e-01, -4.11767783e-01, -1.95295344e-01],
       [-1.18874378e+00,  3.87333211e-01,  1.04379251e-01]])

## Elbow Method

In [6]:
inertia = []

for k in range(1, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(model.inertia_)

In [7]:
import plotly.express as px

elbow_df = pd.DataFrame({
    "Clusters": range(1, 11),
    "Inertia": inertia
})

fig = px.line(
    elbow_df,
    x="Clusters",
    y="Inertia",
    markers=True,
    title="Elbow Method"
)

fig.update_layout(template="plotly_white")

fig.show()

In [8]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans.fit_predict(X_scaled)

rfm.head()

,Customer ID,Recency,Frequency,Monetary,Cluster
0,12346.0,48,1,77183.60,3
1,12347.0,39,2,1187.18,0
2,12348.0,41,2,1120.24,0
3,12350.0,32,1,334.40,0
4,12352.0,5,3,1281.15,0


In [9]:
rfm["Cluster"].value_counts().sort_index()

Cluster
0    1137
1     603
2      22
3       4
Name: count, dtype: int64

In [10]:
cluster_summary = (
    rfm.groupby("Cluster")
       .agg({
           "Recency": "mean",
           "Frequency": "mean",
           "Monetary": "mean"
       })
       .round(2)
)

cluster_summary

,Recency,Frequency,Monetary
Cluster,,,
0,21.97,2.14,851.47
1,77.48,1.23,413.33
2,13.77,17.73,11369.56
3,17.00,5.50,53164.61


## Naming Clusters

In [11]:
cluster_names = {
    0: "Regular Customers",
    1: "At-Risk Customers",
    2: "Loyal High-Value Customers",
    3: "VIP Customers"
}

rfm["Segment"] = rfm["Cluster"].map(cluster_names)

rfm.head()

,Customer ID,Recency,Frequency,Monetary,Cluster,Segment
0,12346.0,48,1,77183.60,3,VIP Customers
1,12347.0,39,2,1187.18,0,Regular Customers
2,12348.0,41,2,1120.24,0,Regular Customers
3,12350.0,32,1,334.40,0,Regular Customers
4,12352.0,5,3,1281.15,0,Regular Customers


In [12]:
rfm.to_csv(
    "../data/processed/customer_segments.csv",
    index=False
)

## Scatter Plot

In [ ]:
fig = px.scatter(
    rfm,
    x="Frequency",
    y="Monetary",
    color="Segment",
    hover_data=["Customer ID", "Recency"],git add ..
    title="Customer Segmentation using RFM Analysis"
)

fig.update_layout(template="plotly_white")

fig.show()